# Data Cleaning Pipeline

---

We already have `pandas`, `json`, and `os` from the previous notebook.  
Here we bring in **`numpy`** — it's a math library that pandas is built on top of.

We need it specifically for `np.where()` in the outlier function,  
which works like an if-else condition applied across an entire column at once.

In [ ]:
import numpy as np
import pandas as pd
import json
import os

# Make sure the processed folder exists before we try to save into it
os.makedirs('data/processed', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("Libraries loaded.")

---

If you're running this notebook standalone (not continuing from the previous one),  
you need to reload the CSV and redefine the logger.  
Skip this cell if `df` and `transformation_log` are already in memory.

In [ ]:
df = pd.read_csv('data/raw/sample_data.csv')

transformation_log = []

def log_transformation(step_name, message):
    entry = {"step": step_name, "message": message}
    transformation_log.append(entry)
    print(f"[LOG] {step_name}: {message}")

print(f"Data reloaded — {len(df)} rows ready for cleaning.")
df.head()

---

This function targets a specific column and fills any missing (`NaN`) values  
with whatever replacement you pass in — defaulting to `'Unknown'`.

**Why do this?**  
Most ML models and aggregations break when they hit `NaN`.  
Filling them keeps the data usable without deleting rows.

- `isnull().sum()` counts how many nulls exist before we fill them  
- `fillna(value)` replaces every `NaN` in the column with the given value  
- The count is logged so we know exactly how many were fixed

In [ ]:
def handle_nulls(dataframe, column, value='Unknown'):
    """Fills null values in a specific column."""
    null_count = dataframe[column].isnull().sum()
    dataframe[column] = dataframe[column].fillna(value)
    log_transformation(
        "Null Handling",
        f"Imputed {null_count} missing values in '{column}' with '{value}'."
    )
    return dataframe

print("handle_nulls() defined.")

---

An outlier is a value that's unreasonably high (or low) compared to the rest.  
Instead of deleting those rows, we **cap** them — clamp the value to a max limit.

**How `np.where()` works here:**
```
np.where(condition, value_if_true, value_if_false)
```
So `np.where(df['value'] > 140, 140, df['value'])` means:  
> *"If the value is above 140, replace it with 140. Otherwise, leave it alone."*

This is applied to every row at once — much faster than looping through each one.

In [ ]:
def cap_outliers(dataframe, column, threshold):
    """Caps values in a column based on a threshold."""
    outliers = dataframe[dataframe[column] > threshold].shape[0]
    dataframe[column] = np.where(
        dataframe[column] > threshold,
        threshold,
        dataframe[column]
    )
    log_transformation(
        "Outlier Detection",
        f"Capped {outliers} values in '{column}' exceeding {threshold}."
    )
    return dataframe

print("cap_outliers() defined.")

---

When a CSV is loaded, date columns usually come in as plain text — just strings like `"2023-01-01"`.  
That means pandas doesn't know they're dates yet, so you can't do things like sort by month or filter by year.

`pd.to_datetime()` converts the column properly so pandas treats each value as an actual date object,  
unlocking time-based operations later.

In [ ]:
def format_dates(dataframe, column):
    """Converts a column to datetime objects."""
    dataframe[column] = pd.to_datetime(dataframe[column])
    log_transformation(
        "Data Formatting",
        f"Converted '{column}' column to datetime objects."
    )
    return dataframe

print("format_dates() defined.")

---

Now we actually apply all three functions one after another.  
Each function takes `df` in, cleans one thing, and returns `df` back — so they chain cleanly.

Order matters here:
1. Fix nulls first (so no `NaN` values interfere with the outlier check)
2. Cap outliers on the `value` column at `140.0`
3. Convert `timestamp` to a proper date type last

In [ ]:
print("Starting cleaning process...\n")

df = handle_nulls(df, 'category', value='Missing')
df = cap_outliers(df, 'value', threshold=140.0)
df = format_dates(df, 'timestamp')

print("\nAll cleaning steps done.")

---

Each function called `log_transformation()` internally,  
so `transformation_log` now has 3 entries — one per cleaning step.

We write the whole list to a `.json` file here.  
This gives you a paper trail of every transformation that ran.

In [ ]:
with open('logs/transformation_log.json', 'w') as f:
    json.dump(transformation_log, f, indent=4)

print("Log saved to: logs/transformation_log.json")
print("\n--- All Log Entries ---")
for i, entry in enumerate(transformation_log):
    print(f"  [{i+1}] {entry['step']}: {entry['message']}")

---

The cleaned dataframe gets saved to a separate folder `data/processed/`  
so the original raw file stays untouched — a good habit in any data project.

We then print a preview with `.head()` to visually confirm the cleaning worked.

In [ ]:
processed_path = 'data/processed/cleaned_data.csv'
df.to_csv(processed_path, index=False)

print(f"Cleaned data saved to: {processed_path}")
print(f"\nFinal shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Nulls remaining: {df.isnull().sum().to_dict()}")

df.head()

---
## Summary

Here's what each function did to the data:

| Function | Column | What it fixed |
|---|---|---|
| `handle_nulls()` | `category` | Replaced `None` values with `'Missing'` |
| `cap_outliers()` | `value` | Clamped anything above `140.0` down to `140.0` |
| `format_dates()` | `timestamp` | Converted string dates to proper datetime objects |

Raw data stays in `data/raw/` — cleaned version lives in `data/processed/`.  
Every change is logged in `logs/transformation_log.json`.